# W3 Homework — A Tool of Your Own

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week03/W3_hw_new_tool.ipynb)

**Goal.** Extend a two-tool agent with a third tool you define — function,
docstring, and the routing tasks that prove the model can find it. The docstring is
the interface (W3 lab); here you write one from nothing and price it with the same
routing score.

The path: setup → the given toolbox and taskset → your tool ✍️ → two routing tasks
for it ✍️ → the measured score → completion.

*Runtime:* ~40 minutes. Due before the W4 session. Reference answers:
`labs/checkpoints/week03/solution.py`, published after the homework deadline.


## 1. Setup

Same standard as the labs: install, paste your key, run the helpers.

*Do:* run the three cells; the last must print `ready`.


In [ ]:
%pip install -q "aisuite[openai,anthropic]"


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"          # alt: "anthropic:claude-haiku-4-5"


In [ ]:
import aisuite

client = aisuite.Client()


def ask(prompt, system=None, temperature=0.0):
    """Single prompt -> reply text."""
    messages = ([{"role": "system", "content": system}] if system else []) + [
        {"role": "user", "content": prompt}]
    response = client.chat.completions.create(model=MODEL, messages=messages,
                                              temperature=temperature)
    return response.choices[0].message.content


print(ask("Reply with exactly: ready"))


## 2. The Given Toolbox and Taskset

Two tools from the lab (with their repaired docstrings) and six routing tasks —
four tool tasks, two no-tool traps. The scorer is the lab's: a task counts when
the expected tool shows in the trace, or, where no trace is recorded, when the
marker appears in the final answer; a no-tool task fails if any tool was called.

*Do:* run both cells; the six-task score should already be high — this is the
baseline your tool must not break.


In [ ]:
PAPER_CATALOG = {
    "react":            {"authors": "Yao et al.", "year": 2022},
    "chain-of-thought": {"authors": "Wei et al.", "year": 2022},
    "toolformer":       {"authors": "Schick et al.", "year": 2023},
    "self-consistency": {"authors": "Wang et al.", "year": 2022},
}


def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression and return the result as a string.
    Use this tool whenever the user asks for a numeric calculation.

    Args:
        expression: The expression to evaluate, in Python arithmetic syntax,
            using only digits, + - * / ( ) and spaces, e.g. "(144 + 6) / 3".
    """
    allowed = set("0123456789+-*/(). ")
    if not expression or not set(expression) <= allowed:
        return f"(error: unsupported characters in {expression!r})"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"(error: {exc})"


def paper_lookup(topic: str) -> str:
    """Look up a paper in the course catalog and return its authors and year.
    Use this tool for questions about the papers covered in this course.

    Args:
        topic: Catalog key. One of: "react", "chain-of-thought", "toolformer",
            "self-consistency".
    """
    entry = PAPER_CATALOG.get(topic.strip().lower())
    if entry is None:
        return f"(unknown topic {topic!r})"
    return f"{entry['authors']}, {entry['year']}"


def called_tools(response):
    """Names of tools recorded in the response trace (empty when none)."""
    names = []
    for step in getattr(response.choices[0], "intermediate_messages", None) or [] or []:
        for call in getattr(step, "tool_calls", None) or []:
            fn = getattr(call, "function", None)
            if fn is not None:
                names.append(getattr(fn, "name", ""))
    return names


In [ ]:
TASKSET = [
    {"request": "What is 17 * 23?", "tool": "calculate", "marker": "391"},
    {"request": "What is (144 + 6) / 3?", "tool": "calculate", "marker": "50"},
    {"request": "In which year was the ReAct paper published, according to the course catalog?",
     "tool": "paper_lookup", "marker": "2022"},
    {"request": "Who are the authors of the Toolformer paper, according to the course catalog?",
     "tool": "paper_lookup", "marker": "Schick"},
    {"request": "Define the term 'function calling' in one sentence.",
     "tool": None, "marker": "function"},
    {"request": "Is 'agent' a French word as well as an English one? Answer briefly.",
     "tool": None, "marker": "French"},
]


def score_routing(taskset, tools):
    """Taskset -> number of correctly routed requests; prints one line per task."""
    correct = 0
    for task in taskset:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": task["request"]}],
            tools=tools, max_turns=5,
        )
        names = called_tools(response)
        text = (response.choices[0].message.content or "")
        if task["tool"] is None:
            ok = not names and task["marker"].lower() in text.lower()
        elif names:
            ok = task["tool"] in names
        else:
            ok = task["marker"].lower() in text.lower()
        correct += ok
        print(f"{'OK ' if ok else 'MISS'} expected={str(task['tool']):>16}  {task['request'][:52]}")
    return correct


baseline_score = score_routing(TASKSET, [calculate, paper_lookup])
print(f"\nbaseline: {baseline_score}/{len(TASKSET)}")


## 3. Your Tool ✍️

The course needs a schedule lookup: given a week number as a string, return that
week's topic. The data is below; the function body is three lines. **The graded
work is the docstring** — the model never sees the body.

Requirements, from the lab's checklist: one sentence saying what the tool does
*and when to use it*; one sentence saying when **not** to use it (paper questions
belong to `paper_lookup`); an `Args:` entry stating the parameter's meaning and
format ("a week number from 1 to 15, digits only, e.g. \"9\"").

Hints: copy the shape of `paper_lookup`'s docstring; name the argument's format
explicitly — "week 9", "9", and "Week09" are what users will type.


In [ ]:
COURSE_SCHEDULE = {
    "1": "What is an Agent?", "2": "Prompting & Reasoning", "3": "Tool Use",
    "4": "The Agent Loop (ReAct)", "5": "Reflection & Evaluation",
    "6": "Multi-Agent Systems", "7": "Planning & Search", "8": "Midterm exam",
    "9": "Retrieval-Augmented Generation", "10": "Context Engineering & Memory",
    "11": "Reasoning Models & RL", "12": "Inference Economics & Benchmarks",
    "13": "Safety & Security", "14": "Final Presentations", "15": "Final exam",
}


### FILL IN (START) ###
def course_schedule(week: str) -> str:
    """Schedule."""
    digits = "".join(ch for ch in str(week) if ch.isdigit())
    return COURSE_SCHEDULE.get(digits, f"(no week {week!r} in the schedule)")
### FILL IN (END) ###

print(course_schedule("9"))
print(course_schedule("week 8"))


## 4. Two Routing Tasks for It ✍️

Add two tasks that exercise your tool: one asking a week's topic, one asking about
the week 8 session. Requirements: the dict shape of `TASKSET`, `"tool":
"course_schedule"`, and a marker word that must appear in a correct answer
("retrieval" for week 9, "midterm" for week 8).

Target: **≥ 7 of 8** on the extended set — and read every MISS before touching
anything: is the miss in your docstring, or in your task's marker?


In [ ]:
### FILL IN (START) ###
MY_TASKS = [
    {"request": "", "tool": "course_schedule", "marker": ""},   # starter — write the request
    {"request": "", "tool": "course_schedule", "marker": ""},
]
### FILL IN (END) ###

FULL_TASKSET = TASKSET + MY_TASKS
extended_score = score_routing(FULL_TASKSET, [calculate, paper_lookup, course_schedule])
TARGET = 7
print(f"\nextended: {extended_score}/{len(FULL_TASKSET)}  (target: {TARGET})")


## 5. Completion Check

Submit: run the notebook top to bottom, then **File → Download → Download .ipynb**
and upload the file to the LMS.


In [ ]:
doc = course_schedule.__doc__ or ""
completion = {
    "docstring says what and when (>= 80 chars)": len(doc.strip()) >= 80,
    "docstring has an Args section": "Args" in doc,
    "docstring states when NOT to use the tool":
        "not" in doc.lower() and "paper" in doc.lower(),
    "two schedule tasks written":
        all(t["request"] and t["marker"] for t in MY_TASKS),
    f"extended score >= {TARGET}/8": extended_score >= TARGET,
}
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nHOMEWORK COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")
